# RHNA & Housing Production

RHNA targets and housing production (applications, entitlements, permits,
completions) for the 18 incorporated jurisdictions in San Diego County
plus the County itself, pulled from HCD, DOF, City of San Diego, and
Census sources.

Years covered, by source:
- APR (applications, entitlements, permits, completions): 2018-2025
- RHNA6: cumulative, 6th Cycle (2021-2029) to date, no year dimension
- DOF, City of SD permits: 2025 only
- ACS: 2020-2024 5-year vintage (its most recent release)

`TARGET_YEAR` (2025) is used throughout for single-year outputs (the main
jurisdiction-year export, regional total). The Power BI export and the
historical APR tables (`production_history`, `applications_history`) span
the full 2018-2025 range.


## Setup

In [ ]:
%pip install pandas numpy requests openpyxl

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import requests
from IPython.display import display


def find_workstream_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").exists():
            return candidate
        my_folder = candidate / "van's work"
        if (my_folder / "notebooks").exists():
            return my_folder
    raise FileNotFoundError(
        "Could not locate workstream root (no notebooks/ folder found nearby)."
    )


ROOT = find_workstream_root()
RAW_DIR = ROOT / "data" / "raw" / "hcd"
PROCESSED_DIR = ROOT / "data" / "processed"
DOCS_DIR = ROOT / "docs"

for d in (RAW_DIR, PROCESSED_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)

from datetime import date
DOWNLOAD_DATE = date.today().isoformat()

print("Workstream root:", ROOT)
print("Download date:", DOWNLOAD_DATE)


In [ ]:
TARGET_YEAR = 2025
ACS_VINTAGE_LABEL = "2020-2024"
ACS_DATA_YEAR = 2024

SAN_DIEGO_CITIES = [
    "Carlsbad", "Chula Vista", "Coronado", "Del Mar", "El Cajon",
    "Encinitas", "Escondido", "Imperial Beach", "La Mesa", "Lemon Grove",
    "National City", "Oceanside", "Poway", "San Diego", "San Marcos",
    "Santee", "Solana Beach", "Vista",
]
COUNTY_JURISDICTION_NAME = "San Diego County"


In [ ]:
COUNTY_NAME_VARIANTS = {
    "san diego county", "county of san diego", "s d county",
    "county san diego", "unincorporated", "unincorporated san diego county",
}


def normalize_jurisdiction(name: object) -> str:
    s = str(name).strip().lower()
    if s == "national city":
        return "national city"
    if s in COUNTY_NAME_VARIANTS:
        return "unincorporated san diego county"
    s = re.sub(r"^city\s+of\s+", "", s)
    s = re.sub(r"^county\s+of\s+", "county ", s)
    s = re.sub(r"\s+city$", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


sd_jur_keys = {normalize_jurisdiction(c) for c in SAN_DIEGO_CITIES}
sd_jur_keys.add(normalize_jurisdiction(COUNTY_JURISDICTION_NAME))


def get_json(url: str, params: dict | None = None, timeout: int = 60):
    response = requests.get(
        url, params=params, timeout=timeout,
        headers={"User-Agent": "CHPD Housing Dashboard Data Validation"},
    )
    if not response.ok:
        raise RuntimeError(f"Request failed ({response.status_code}): {response.url}")
    return response.json()


def find_resource_download_url(package_json: dict, name_contains: str):
    resources = package_json["result"]["resources"]
    matches = [
        r for r in resources
        if name_contains.lower() in r.get("name", "").lower()
        and r.get("format", "").upper() == "CSV"
    ]
    if not matches:
        raise ValueError(f"No CSV resource matching '{name_contains}' found.")
    matches.sort(key=lambda r: r.get("last_modified", ""), reverse=True)
    return matches[0]["url"], matches[0]["name"]


def fill_missing_jurisdiction_years(df: pd.DataFrame, years: list) -> pd.DataFrame:
    """
    Guarantee every (jurisdiction, year) combination exists, filling 0 where
    a jurisdiction reported nothing that year - groupby silently omits
    combinations with zero rows, which reads as a missing/blank value in
    Power BI instead of a real, meaningful zero.
    """
    full_index = pd.MultiIndex.from_product([sorted(sd_jur_keys), years], names=["jur_clean", "year"])
    filled = df.set_index(["jur_clean", "year"]).reindex(full_index).reset_index()
    fill_cols = [c for c in filled.select_dtypes(include="number").columns if "share" not in c and "pct" not in c]
    filled[fill_cols] = filled[fill_cols].fillna(0)
    return filled


## APR (permits, entitlements, completions)

In [ ]:
CKAN_PACKAGE_URL = (
    "https://data.ca.gov/api/3/action/package_show"
    "?id=housing-element-annual-progress-report-apr-data-by-jurisdiction-and-year"
)
package_json = get_json(CKAN_PACKAGE_URL)
table_a2_url, table_a2_name = find_resource_download_url(package_json, "Table A2")

raw_path = RAW_DIR / "apr_table_a2_raw.csv"
if not raw_path.exists():
    resp = requests.get(table_a2_url, timeout=300)
    resp.raise_for_status()
    raw_path.write_bytes(resp.content)

apr_raw = pd.read_csv(raw_path, low_memory=False)
print(apr_raw.shape)
apr_raw.head()


In [ ]:
print(apr_raw.columns.tolist())

**Development stages tracked here, kept strictly separate:**

| Stage | Source | Metric |
|---|---|---|
| Application submitted | APR Table A | `application_units_total` |
| Entitlement | APR Table A2 (unprefixed `*_INCOME_*` columns; cross-checked against HCD's own auto-populated `NO_ENTITLEMENTS` total) | `ent_units_total` |
| Building permit | APR Table A2 (`BP_*_INCOME` columns; cross-checked against `NO_BUILDING_PERMITS`) | `bp_units_total` |
| Completion (certificate of occupancy) | APR Table A2 (`CO_*_INCOME` columns; cross-checked against `NO_OTHER_FORMS_OF_READINESS`) | `co_units_total` |

**Units under construction** is the one stage from the dashboard reorg
guidance that is **not tracked anywhere in HCD's APR data** - no table
captures this milestone.

Never summed into one another - a project can appear as an
application one year, entitled the next, permitted the year after,
so combining stages double-counts the same units.

In [ ]:
JURISDICTION_COL = "JURIS_NAME"
YEAR_COL = "YEAR"

apr_raw["jur_clean"] = apr_raw[JURISDICTION_COL].map(normalize_jurisdiction)
apr_raw[YEAR_COL] = pd.to_numeric(apr_raw[YEAR_COL], errors="coerce")

sd_apr = apr_raw[apr_raw["jur_clean"].isin(sd_jur_keys)].copy()
sd_apr_target_year = sd_apr[sd_apr[YEAR_COL] == TARGET_YEAR].copy()

print("SD rows, all years:", len(sd_apr))
print(f"SD rows, {TARGET_YEAR}:", len(sd_apr_target_year))
missing = sd_jur_keys - set(sd_apr_target_year["jur_clean"].unique())
if missing:
    print(f"No {TARGET_YEAR} row yet:", sorted(missing))


### Duplicate check

Never verified whether the same project appears more than once in the raw
file - a duplicate row would silently inflate every total below (permits,
completions, entitlements) without any error or warning.

In [ ]:
# Exact duplicate rows (every column identical).
exact_dupes = sd_apr_target_year.duplicated().sum()
print("Exact duplicate rows:", exact_dupes)

# Same tracking ID reported more than once in the same jurisdiction-year --
# a legitimate multi-phase project could share an APN across years, but
# the same JURS_TRACKING_ID + year combination should be unique.
id_dupes = sd_apr_target_year[sd_apr_target_year["JURS_TRACKING_ID"].notna()].duplicated(
    subset=["jur_clean", "YEAR", "JURS_TRACKING_ID"], keep=False
)
print("Rows sharing a JURS_TRACKING_ID within the same jurisdiction-year:", id_dupes.sum())
if id_dupes.sum() > 0:
    display(sd_apr_target_year[id_dupes][["jur_clean", "YEAR", "JURS_TRACKING_ID", "APN", "PROJECT_NAME"]])


In [ ]:
# Table A2 tracks THREE separate stages, not two: unprefixed *_INCOME_*
# columns are entitlement-stage units (paired with ENT_APPROVE_DT1), BP_*
# are building permits, CO_* are completions. Kept as three fully separate
# metrics - never summed into one another.
#
# Same 4-tier grouping used for RHNA (very_low/low/moderate/above_moderate)
# - rolls Acutely Low + Extremely Low + Very Low columns into "very_low",
# matching HCD's own convention that those sub-tiers count toward
# very-low-income reporting.
TIER_SUFFIX_GROUPS = {
    "very_low": [
        "ACUTELY_LOW_INCOME_DR", "ACUTELY_LOW_INCOME_NDR",
        "EXTREMELY_LOW_INCOME_DR", "EXTREMELY_LOW_INCOME_NDR",
        "VLOW_INCOME_DR", "VLOW_INCOME_NDR",
    ],
    "low": ["LOW_INCOME_DR", "LOW_INCOME_NDR"],
    "moderate": ["MOD_INCOME_DR", "MOD_INCOME_NDR"],
    "above_moderate": ["ABOVE_MOD_INCOME"],
}
ALL_TIER_SUFFIXES = [s for suffixes in TIER_SUFFIX_GROUPS.values() for s in suffixes]

# Built explicitly from the known tier structure, not a loose "contains
# INCOME" text search - Table A2 also has EXTR_LOW_INCOME_UNITS, an
# unrelated field that a loose search would incorrectly sweep in.
ENT_INCOME_COLS = [s for s in ALL_TIER_SUFFIXES if s in sd_apr_target_year.columns]
BP_INCOME_COLS = [f"BP_{s}" for s in ALL_TIER_SUFFIXES if f"BP_{s}" in sd_apr_target_year.columns]
CO_INCOME_COLS = [f"CO_{s}" for s in ALL_TIER_SUFFIXES if f"CO_{s}" in sd_apr_target_year.columns]

sd_apr_target_year["ent_units_row"] = sd_apr_target_year[ENT_INCOME_COLS].sum(axis=1, numeric_only=True)
sd_apr_target_year["bp_units_row"] = sd_apr_target_year[BP_INCOME_COLS].sum(axis=1, numeric_only=True)
sd_apr_target_year["co_units_row"] = sd_apr_target_year[CO_INCOME_COLS].sum(axis=1, numeric_only=True)

above_mod_ent = [c for c in ENT_INCOME_COLS if "ABOVE" in c.upper()]
above_mod_bp = [c for c in BP_INCOME_COLS if "ABOVE" in c.upper()]
above_mod_co = [c for c in CO_INCOME_COLS if "ABOVE" in c.upper()]

sd_apr_target_year["ent_affordable_row"] = (
    sd_apr_target_year["ent_units_row"] - sd_apr_target_year[above_mod_ent].sum(axis=1, numeric_only=True)
)
sd_apr_target_year["bp_affordable_row"] = (
    sd_apr_target_year["bp_units_row"] - sd_apr_target_year[above_mod_bp].sum(axis=1, numeric_only=True)
)
sd_apr_target_year["co_affordable_row"] = (
    sd_apr_target_year["co_units_row"] - sd_apr_target_year[above_mod_co].sum(axis=1, numeric_only=True)
)

# Per-tier entitlement, permit, and completion columns.
tier_agg_kwargs = {}
for tier, suffixes in TIER_SUFFIX_GROUPS.items():
    ent_cols = [s for s in suffixes if s in sd_apr_target_year.columns]
    bp_cols = [f"BP_{s}" for s in suffixes if f"BP_{s}" in sd_apr_target_year.columns]
    co_cols = [f"CO_{s}" for s in suffixes if f"CO_{s}" in sd_apr_target_year.columns]
    sd_apr_target_year[f"ent_{tier}_row"] = sd_apr_target_year[ent_cols].sum(axis=1, numeric_only=True)
    sd_apr_target_year[f"bp_{tier}_row"] = sd_apr_target_year[bp_cols].sum(axis=1, numeric_only=True)
    sd_apr_target_year[f"co_{tier}_row"] = sd_apr_target_year[co_cols].sum(axis=1, numeric_only=True)
    tier_agg_kwargs[f"ent_{tier}_total"] = (f"ent_{tier}_row", "sum")
    tier_agg_kwargs[f"bp_{tier}_total"] = (f"bp_{tier}_row", "sum")
    tier_agg_kwargs[f"co_{tier}_total"] = (f"co_{tier}_row", "sum")

# The raw file is row-level (one row per project/address) - group up to
# jurisdiction-year before this becomes a usable production metric.
production_by_year = (
    sd_apr_target_year
    .groupby("jur_clean", as_index=False)
    .agg(
        year=(YEAR_COL, "first"),
        ent_units_total=("ent_units_row", "sum"),
        bp_units_total=("bp_units_row", "sum"),
        co_units_total=("co_units_row", "sum"),
        ent_affordable_total=("ent_affordable_row", "sum"),
        bp_affordable_total=("bp_affordable_row", "sum"),
        co_affordable_total=("co_affordable_row", "sum"),
        project_rows=("jur_clean", "size"),
        **tier_agg_kwargs,
    )
)
production_by_year["ent_affordable_share"] = production_by_year["ent_affordable_total"] / production_by_year["ent_units_total"]
production_by_year["bp_affordable_share"] = production_by_year["bp_affordable_total"] / production_by_year["bp_units_total"]
production_by_year["co_affordable_share"] = production_by_year["co_affordable_total"] / production_by_year["co_units_total"]

# Sanity check: per-tier ent/bp/co columns should sum back to the overall total.
tier_ent_sum = production_by_year[[f"ent_{t}_total" for t in TIER_SUFFIX_GROUPS]].sum(axis=1)
tier_bp_sum = production_by_year[[f"bp_{t}_total" for t in TIER_SUFFIX_GROUPS]].sum(axis=1)
tier_co_sum = production_by_year[[f"co_{t}_total" for t in TIER_SUFFIX_GROUPS]].sum(axis=1)
assert (tier_ent_sum == production_by_year["ent_units_total"]).all(), "ENT tier columns do not sum to ent_units_total"
assert (tier_bp_sum == production_by_year["bp_units_total"]).all(), "BP tier columns do not sum to bp_units_total"
assert (tier_co_sum == production_by_year["co_units_total"]).all(), "CO tier columns do not sum to co_units_total"

production_by_year


### Cross-check against HCD's auto-populated totals

`NO_ENTITLEMENTS` / `NO_BUILDING_PERMITS` / `NO_OTHER_FORMS_OF_READINESS`
mean "Number Of," not a flag - HCD auto-populates these from the same
per-tier income columns summed below. Used here as an independent check
on ent/bp/co totals.

In [ ]:
validation_cols = {
    "ent_units_total": "NO_ENTITLEMENTS",
    "bp_units_total": "NO_BUILDING_PERMITS",
    "co_units_total": "NO_OTHER_FORMS_OF_READINESS",
}

hcd_totals = sd_apr_target_year.groupby("jur_clean", as_index=False)[list(validation_cols.values())].sum()
stage_check = production_by_year.merge(hcd_totals, on="jur_clean")

for our_col, hcd_col in validation_cols.items():
    stage_check[f"{our_col}_diff"] = stage_check[our_col] - stage_check[hcd_col]

diff_cols = [f"{c}_diff" for c in validation_cols]
print("Max absolute difference per stage:")
print(stage_check[diff_cols].abs().max())
stage_check[["jur_clean"] + list(validation_cols.keys()) + list(validation_cols.values()) + diff_cols]


## Production by housing type

Distinct from the income-tier breakdown above - this splits permits and
completions by structure type (`UNIT_CAT`), per the dashboard reorg
guidance's "Production by Housing Type" section. Category values are
verified from the real data below, not assumed.

In [ ]:
print(sd_apr_target_year["UNIT_CAT"].value_counts(dropna=False))


In [ ]:
# TODO: confirm this mapping against the value_counts() output above --
# HCD's standard Table A2 categories, adjust left-hand keys if the real
# values differ.
UNIT_CAT_LABELS = {
    "SFD": "Single-Family Detached",
    "SFA": "Single-Family Attached",
    "2 to 4": "2-4 Unit Buildings",
    "5+": "5+ Unit Buildings",
    "ADU": "Accessory Dwelling Unit",
    "MH": "Mobile / Manufactured Home",
}

production_by_type = (
    sd_apr_target_year
    .groupby(["jur_clean", "UNIT_CAT"], as_index=False)
    .agg(
        bp_units_total=("bp_units_row", "sum"),
        co_units_total=("co_units_row", "sum"),
        project_rows=("jur_clean", "size"),
    )
)
production_by_type["housing_type"] = production_by_type["UNIT_CAT"].map(UNIT_CAT_LABELS).fillna(production_by_type["UNIT_CAT"])

# Sanity check: summed across types should equal the overall total per jurisdiction.
check = production_by_type.groupby("jur_clean")[["bp_units_total", "co_units_total"]].sum()
original = production_by_year.set_index("jur_clean")[["bp_units_total", "co_units_total"]]
assert (check.reindex(original.index) == original).all().all(), "Type breakdown does not sum to overall totals"

print(production_by_type.shape)
production_by_type.sort_values(["jur_clean", "bp_units_total"], ascending=[True, False])


In [ ]:
type_output_path = PROCESSED_DIR / f"apr_production_by_housing_type_{TARGET_YEAR}.csv"
production_by_type.to_csv(type_output_path, index=False)
print("Saved:", type_output_path)


### Historical range check (2018-2025)

APR data collection began in 2018. Confirms every year is actually
present for San Diego County, not just assumed.

In [ ]:
APR_START_YEAR = 2018
APR_YEARS = list(range(APR_START_YEAR, TARGET_YEAR + 1))

ENT_INCOME_COLS_ALL = [s for s in ALL_TIER_SUFFIXES if s in sd_apr.columns]
BP_INCOME_COLS_ALL = [f"BP_{s}" for s in ALL_TIER_SUFFIXES if f"BP_{s}" in sd_apr.columns]
CO_INCOME_COLS_ALL = [f"CO_{s}" for s in ALL_TIER_SUFFIXES if f"CO_{s}" in sd_apr.columns]

sd_apr["ent_units_row"] = sd_apr[ENT_INCOME_COLS_ALL].sum(axis=1, numeric_only=True)
sd_apr["bp_units_row"] = sd_apr[BP_INCOME_COLS_ALL].sum(axis=1, numeric_only=True)
sd_apr["co_units_row"] = sd_apr[CO_INCOME_COLS_ALL].sum(axis=1, numeric_only=True)

above_mod_ent_all = [c for c in ENT_INCOME_COLS_ALL if "ABOVE" in c.upper()]
above_mod_bp_all = [c for c in BP_INCOME_COLS_ALL if "ABOVE" in c.upper()]
above_mod_co_all = [c for c in CO_INCOME_COLS_ALL if "ABOVE" in c.upper()]

sd_apr["ent_affordable_row"] = sd_apr["ent_units_row"] - sd_apr[above_mod_ent_all].sum(axis=1, numeric_only=True)
sd_apr["bp_affordable_row"] = sd_apr["bp_units_row"] - sd_apr[above_mod_bp_all].sum(axis=1, numeric_only=True)
sd_apr["co_affordable_row"] = sd_apr["co_units_row"] - sd_apr[above_mod_co_all].sum(axis=1, numeric_only=True)

tier_agg_kwargs_hist = {}
for tier, suffixes in TIER_SUFFIX_GROUPS.items():
    ent_cols = [s for s in suffixes if s in sd_apr.columns]
    bp_cols = [f"BP_{s}" for s in suffixes if f"BP_{s}" in sd_apr.columns]
    co_cols = [f"CO_{s}" for s in suffixes if f"CO_{s}" in sd_apr.columns]
    sd_apr[f"ent_{tier}_row"] = sd_apr[ent_cols].sum(axis=1, numeric_only=True)
    sd_apr[f"bp_{tier}_row"] = sd_apr[bp_cols].sum(axis=1, numeric_only=True)
    sd_apr[f"co_{tier}_row"] = sd_apr[co_cols].sum(axis=1, numeric_only=True)
    tier_agg_kwargs_hist[f"ent_{tier}_total"] = (f"ent_{tier}_row", "sum")
    tier_agg_kwargs_hist[f"bp_{tier}_total"] = (f"bp_{tier}_row", "sum")
    tier_agg_kwargs_hist[f"co_{tier}_total"] = (f"co_{tier}_row", "sum")

production_history = (
    sd_apr[sd_apr[YEAR_COL].isin(APR_YEARS)]
    .groupby(["jur_clean", YEAR_COL], as_index=False)
    .agg(
        ent_units_total=("ent_units_row", "sum"),
        bp_units_total=("bp_units_row", "sum"),
        co_units_total=("co_units_row", "sum"),
        ent_affordable_total=("ent_affordable_row", "sum"),
        bp_affordable_total=("bp_affordable_row", "sum"),
        co_affordable_total=("co_affordable_row", "sum"),
        project_rows=("jur_clean", "size"),
        **tier_agg_kwargs_hist,
    )
    .rename(columns={YEAR_COL: "year"})
)
production_history["ent_affordable_share"] = production_history["ent_affordable_total"] / production_history["ent_units_total"]
production_history["bp_affordable_share"] = production_history["bp_affordable_total"] / production_history["bp_units_total"]
production_history["co_affordable_share"] = production_history["co_affordable_total"] / production_history["co_units_total"]

print(production_history.shape)
production_history.head()


In [ ]:
years_present = sorted(production_history["year"].dropna().unique())
years_missing = sorted(set(APR_YEARS) - set(years_present))
print("Years present:", years_present)
if years_missing:
    print("Years missing entirely from the pull:", years_missing)

coverage = (
    production_history
    .groupby("year")["jur_clean"]
    .agg(jurisdictions="nunique", rows="count")
    .reindex(APR_YEARS)
)
coverage["missing_jurisdictions"] = coverage["jurisdictions"].apply(
    lambda n: 19 - n if pd.notna(n) else 19
)
coverage


In [ ]:
# Reindex AFTER the coverage check above, so the check reflects real
# reporting gaps - this fill is only for downstream use (Power BI export).
production_history = fill_missing_jurisdiction_years(production_history, APR_YEARS)
print(production_history.shape)


In [ ]:
history_output_path = PROCESSED_DIR / f"apr_production_{APR_START_YEAR}_{TARGET_YEAR}_by_jurisdiction_year.csv"
production_history.to_csv(history_output_path, index=False)
print("Saved:", history_output_path)


## APR Table A (applications)

Applications live in a separate table from entitlements/permits/completions
(Table A vs. Table A2). Loaded the same way as Table A2 - CKAN action API,
resolved by exact resource name.

In [ ]:
table_a_package_json = get_json(CKAN_PACKAGE_URL)

table_a_matches = [
    r for r in table_a_package_json["result"]["resources"]
    if r.get("name", "").strip().lower() == "apr table a"
    and r.get("format", "").upper() == "CSV"
]
if not table_a_matches:
    raise ValueError("No exact 'APR Table A' CSV resource found in the package.")
table_a_url = table_a_matches[0]["url"]
table_a_name = table_a_matches[0]["name"]
print("Using resource:", table_a_name)
print("Download URL:", table_a_url)

table_a_raw_path = RAW_DIR / "apr_table_a_raw.csv"
if not table_a_raw_path.exists():
    resp = requests.get(table_a_url, timeout=300)
    resp.raise_for_status()
    table_a_raw_path.write_bytes(resp.content)

apr_table_a_raw = pd.read_csv(table_a_raw_path, low_memory=False)
print(apr_table_a_raw.shape)
apr_table_a_raw.head()


In [ ]:
print(apr_table_a_raw.columns.tolist())

In [ ]:
TABLE_A_JUR_COL = "JURIS_NAME"
TABLE_A_YEAR_COL = "YEAR"

apr_table_a_raw["jur_clean"] = apr_table_a_raw[TABLE_A_JUR_COL].map(normalize_jurisdiction)
apr_table_a_raw[TABLE_A_YEAR_COL] = pd.to_numeric(apr_table_a_raw[TABLE_A_YEAR_COL], errors="coerce")

sd_apr_table_a = apr_table_a_raw[apr_table_a_raw["jur_clean"].isin(sd_jur_keys)].copy()
sd_apr_table_a_target_year = sd_apr_table_a[sd_apr_table_a[TABLE_A_YEAR_COL] == TARGET_YEAR].copy()

print("SD rows, all years:", len(sd_apr_table_a))
print(f"SD rows, {TARGET_YEAR}:", len(sd_apr_table_a_target_year))
missing = sd_jur_keys - set(sd_apr_table_a_target_year["jur_clean"].unique())
if missing:
    print(f"No {TARGET_YEAR} row yet:", sorted(missing))


In [ ]:
APPLICATION_INCOME_COLS = [
    "ACUTELY_LOW_INCOME_DR", "ACUTELY_LOW_INCOME_NDR",
    "EXTREMELY_LOW_INCOME_DR", "EXTREMELY_LOW_INCOME_NDR",
    "VLOW_INCOME_DR", "VLOW_INCOME_NDR",
    "LOW_INCOME_DR", "LOW_INCOME_NDR",
    "MOD_INCOME_DR", "MOD_INCOME_NDR",
    "ABOVE_MOD_INCOME",
]
above_mod_application = [c for c in APPLICATION_INCOME_COLS if "ABOVE" in c.upper()]

sd_apr_table_a_target_year["application_units_row"] = (
    sd_apr_table_a_target_year[APPLICATION_INCOME_COLS].sum(axis=1, numeric_only=True)
)
sd_apr_table_a_target_year["application_affordable_row"] = (
    sd_apr_table_a_target_year["application_units_row"]
    - sd_apr_table_a_target_year[above_mod_application].sum(axis=1, numeric_only=True)
)

application_tier_agg_kwargs = {}
for tier, suffixes in TIER_SUFFIX_GROUPS.items():
    app_cols = [s for s in suffixes if s in sd_apr_table_a_target_year.columns]
    sd_apr_table_a_target_year[f"application_{tier}_row"] = (
        sd_apr_table_a_target_year[app_cols].sum(axis=1, numeric_only=True)
    )
    application_tier_agg_kwargs[f"application_{tier}_total"] = (f"application_{tier}_row", "sum")

applications_by_jurisdiction = (
    sd_apr_table_a_target_year
    .groupby("jur_clean", as_index=False)
    .agg(
        application_units_total=("application_units_row", "sum"),
        application_affordable_total=("application_affordable_row", "sum"),
        application_rows=("jur_clean", "size"),
        **application_tier_agg_kwargs,
    )
)
applications_by_jurisdiction["application_affordable_share"] = (
    applications_by_jurisdiction["application_affordable_total"] / applications_by_jurisdiction["application_units_total"]
)

print(applications_by_jurisdiction.shape)
applications_by_jurisdiction


In [ ]:
proposed_check = sd_apr_table_a_target_year.groupby("jur_clean", as_index=False).agg(
    tot_proposed_units=("TOT_PROPOSED_UNITS", "sum")
)
check = applications_by_jurisdiction.merge(proposed_check, on="jur_clean")
check["diff"] = check["application_units_total"] - check["tot_proposed_units"]
check[["jur_clean", "application_units_total", "tot_proposed_units", "diff"]]


### Historical applications (2018-2025)

Same treatment as APR Table A2 - `sd_apr_table_a` already has all years
for San Diego County; aggregate the full range rather than just
`TARGET_YEAR`.

In [ ]:
sd_apr_table_a["application_units_row"] = (
    sd_apr_table_a[APPLICATION_INCOME_COLS].sum(axis=1, numeric_only=True)
)
sd_apr_table_a["application_affordable_row"] = (
    sd_apr_table_a["application_units_row"] - sd_apr_table_a[above_mod_application].sum(axis=1, numeric_only=True)
)

application_tier_agg_kwargs_hist = {}
for tier, suffixes in TIER_SUFFIX_GROUPS.items():
    app_cols = [s for s in suffixes if s in sd_apr_table_a.columns]
    sd_apr_table_a[f"application_{tier}_row"] = sd_apr_table_a[app_cols].sum(axis=1, numeric_only=True)
    application_tier_agg_kwargs_hist[f"application_{tier}_total"] = (f"application_{tier}_row", "sum")

applications_history = (
    sd_apr_table_a[sd_apr_table_a[TABLE_A_YEAR_COL].isin(APR_YEARS)]
    .groupby(["jur_clean", TABLE_A_YEAR_COL], as_index=False)
    .agg(
        application_units_total=("application_units_row", "sum"),
        application_affordable_total=("application_affordable_row", "sum"),
        application_rows=("jur_clean", "size"),
        **application_tier_agg_kwargs_hist,
    )
    .rename(columns={TABLE_A_YEAR_COL: "year"})
)
applications_history["application_affordable_share"] = (
    applications_history["application_affordable_total"] / applications_history["application_units_total"]
)

years_present_app = sorted(applications_history["year"].dropna().unique())
years_missing_app = sorted(set(APR_YEARS) - set(years_present_app))
print("Years present (applications):", years_present_app)
if years_missing_app:
    print("Years missing entirely:", years_missing_app)

coverage_app = (
    applications_history
    .groupby("year")["jur_clean"]
    .agg(jurisdictions="nunique", rows="count")
    .reindex(APR_YEARS)
)
coverage_app["missing_jurisdictions"] = coverage_app["jurisdictions"].apply(
    lambda n: 19 - n if pd.notna(n) else 19
)
print(applications_history.shape)
coverage_app


In [ ]:
for yr in [2018, 2019, 2020]:
    present = set(applications_history[applications_history["year"] == yr]["jur_clean"])
    missing = sd_jur_keys - present
    if missing:
        print(f"{yr}: missing {sorted(missing)}")


In [ ]:
applications_history = fill_missing_jurisdiction_years(applications_history, APR_YEARS)
print(applications_history.shape)


## RHNA 6th Cycle targets

In [ ]:
RHNA_PACKAGE_URL = "https://data.ca.gov/api/3/action/package_show?id=rhna-progress-report"
rhna_package_json = get_json(RHNA_PACKAGE_URL)
rhna6_url, rhna6_name = find_resource_download_url(rhna_package_json, "6th Cycle RHNA Progress Report")

rhna_raw_path = RAW_DIR / "rhna6_progress_raw.csv"
if not rhna_raw_path.exists():
    resp = requests.get(rhna6_url, timeout=120)
    resp.raise_for_status()
    rhna_raw_path.write_bytes(resp.content)

rhna_raw = pd.read_csv(rhna_raw_path, low_memory=False)
print(rhna_raw.shape)
rhna_raw.head()


In [ ]:
print(rhna_raw.columns.tolist())

**RHNA progress basis:** `rhna_reported_<tier>` / `rhna_pct_achieved_<tier>`
/ `rhna_remaining_<tier>` are based on **building permits issued**, per
HCD's own RHNA-credit methodology - not completions, not entitlements.
Distinct from `co_units_total` (completions); don't conflate the two.

In [ ]:
RHNA_JUR_COL = "Jurisdiction"

rhna_raw["jur_clean"] = rhna_raw[RHNA_JUR_COL].map(normalize_jurisdiction)
sd_rhna6 = rhna_raw[rhna_raw["jur_clean"].isin(sd_jur_keys)].copy()

print("SD rows:", len(sd_rhna6))
missing = sd_jur_keys - set(sd_rhna6["jur_clean"].unique())
if missing:
    print("Not found:", sorted(missing))
sd_rhna6.head()


## CA DOF population & housing estimates (E-5)

In [ ]:
DOF_RAW_PATH = RAW_DIR.parent / "dof" / "e5_population_housing.xlsx"
DOF_SHEET_NAME = f"E5CityCounty{TARGET_YEAR}"

dof_raw = pd.read_excel(DOF_RAW_PATH, sheet_name=DOF_SHEET_NAME, header=3)
dof_raw = dof_raw.rename(columns={"County/City/State": "name"})
dof_raw["name"] = dof_raw["name"].astype(str).str.strip()

# The raw sheet has two columns both literally named "Total" - population
# total and housing-unit total. pandas auto-disambiguates the second one
# to "Total.1" on read; renamed here to something unambiguous.
dof_raw = dof_raw.rename(columns={"Total": "population_total", "Total.1": "housing_units_total"})

dof_raw["is_county_header"] = dof_raw["population_total"].isna() & dof_raw["name"].str.contains("County", na=False)
dof_raw["county"] = dof_raw["name"].where(dof_raw["is_county_header"]).ffill()

sd_dof = dof_raw[
    (dof_raw["county"] == "San Diego County")
    & (~dof_raw["is_county_header"])
    & (dof_raw["population_total"].notna())
    & (~dof_raw["name"].isin(["Incorporated", "County Total"]))
].copy()

sd_dof["jur_clean"] = sd_dof["name"].map(
    lambda n: normalize_jurisdiction(COUNTY_JURISDICTION_NAME) if n.strip() == "Unincorporated" else normalize_jurisdiction(n)
)
sd_dof["year"] = TARGET_YEAR

sd_dof["vacant_units"] = sd_dof["housing_units_total"] - sd_dof["Occupied"]
sd_dof["single_family_units"] = sd_dof["Single Detached"] + sd_dof["Single Attached"]
sd_dof["multifamily_units"] = sd_dof["Two to Four"] + sd_dof["Five Plus"]
sd_dof["mobile_home_units"] = sd_dof["Mobile Homes"]

structure_sum = sd_dof["single_family_units"] + sd_dof["multifamily_units"] + sd_dof["mobile_home_units"]
assert (structure_sum == sd_dof["housing_units_total"]).all(), "Structure-type columns do not sum to housing_units_total"

print("SD rows:", len(sd_dof))
missing = sd_jur_keys - set(sd_dof["jur_clean"].unique())
if missing:
    print("Not found:", sorted(missing))
sd_dof[[
    "name", "population_total", "housing_units_total", "Occupied", "vacant_units",
    "single_family_units", "multifamily_units", "mobile_home_units",
]]


## Census ACS (2020-2024 5-year estimates)

In [ ]:
from getpass import getpass

CALIFORNIA_STATE_FIPS = "06"
SAN_DIEGO_COUNTY_FIPS = "073"

CENSUS_API_KEY = getpass("Census API key: ").strip()
if not CENSUS_API_KEY:
    raise ValueError("A Census API key is required.")

ACS_VARS = {
    "B01003_001E": "population_total",
    "B25001_001E": "housing_units_total",
    "B25003_002E": "owner_occupied",
    "B25003_003E": "renter_occupied",
}

acs_url = f"https://api.census.gov/data/{ACS_DATA_YEAR}/acs/acs5"
params = {
    "get": ",".join(["NAME", *ACS_VARS.keys()]),
    "for": "place:*",
    "in": f"state:{CALIFORNIA_STATE_FIPS}",
    "key": CENSUS_API_KEY,
}

acs_raw = get_json(acs_url, params=params)
acs_df = pd.DataFrame(acs_raw[1:], columns=acs_raw[0]).rename(columns=ACS_VARS)
acs_df["jur_clean"] = acs_df["NAME"].str.replace(r" city, California$", "", regex=True).map(normalize_jurisdiction)

sd_acs = acs_df[acs_df["jur_clean"].isin(sd_jur_keys)].copy()
print(f"SD rows ({ACS_VINTAGE_LABEL}):", len(sd_acs))
sd_acs.head()


In [ ]:
acs_output_path = PROCESSED_DIR / f"sd_acs_{ACS_DATA_YEAR}_by_jurisdiction.csv"
sd_acs.to_csv(acs_output_path, index=False)
print("Saved:", acs_output_path)


### 2021 ACS 5-year estimates (2017-2021 vintage)

A separate pull, not derivable from the 2024 vintage above - ACS 5-year
estimates are rolling averages, not point-in-time counts. "2021 ACS" means
the 2017-2021 window blended together; "2024 ACS" means 2020-2024 blended
together. They overlap by only one year and come from different survey
respondents, so one cannot be computed from the other.

In [ ]:
ACS_2021_DATA_YEAR = 2021
ACS_2021_VINTAGE_LABEL = "2017-2021"

acs_2021_url = f"https://api.census.gov/data/{ACS_2021_DATA_YEAR}/acs/acs5"
params_2021 = {
    "get": ",".join(["NAME", *ACS_VARS.keys()]),
    "for": "place:*",
    "in": f"state:{CALIFORNIA_STATE_FIPS}",
    "key": CENSUS_API_KEY,
}

acs_2021_raw = get_json(acs_2021_url, params=params_2021)
acs_2021_df = pd.DataFrame(acs_2021_raw[1:], columns=acs_2021_raw[0]).rename(columns=ACS_VARS)
acs_2021_df["jur_clean"] = acs_2021_df["NAME"].str.replace(r" city, California$", "", regex=True).map(normalize_jurisdiction)

sd_acs_2021 = acs_2021_df[acs_2021_df["jur_clean"].isin(sd_jur_keys)].copy()
print(f"SD rows ({ACS_2021_VINTAGE_LABEL}):", len(sd_acs_2021))
sd_acs_2021.head()


In [ ]:
acs_2021_output_path = PROCESSED_DIR / f"sd_acs_{ACS_2021_DATA_YEAR}_by_jurisdiction.csv"
sd_acs_2021.to_csv(acs_2021_output_path, index=False)
print("Saved:", acs_2021_output_path)


## Decennial Census 2000 and 2010 (housing stock benchmarks)

Different data product from ACS - Summary File 1 (SF1), the actual full
count, not a survey sample. Variable `H001001` is "Total Housing Units,"
the same code across both Census years. Point-in-time counts (as of April
1 of that year), not rolling averages like the ACS pulls above.

Not yet run against the live API in this pass - printing the raw response
first to confirm the variable/geography structure before trusting it, same
as every other new source in this notebook.

In [ ]:
census_2000_url = "https://api.census.gov/data/2000/dec/sf1"
params_2000 = {
    "get": "NAME,H001001",
    "for": "place:*",
    "in": f"state:{CALIFORNIA_STATE_FIPS}",
    "key": CENSUS_API_KEY,
}

census_2000_raw = get_json(census_2000_url, params=params_2000)
census_2000_df = pd.DataFrame(census_2000_raw[1:], columns=census_2000_raw[0])
census_2000_df = census_2000_df.rename(columns={"H001001": "housing_units_total"})
census_2000_df["housing_units_total"] = pd.to_numeric(census_2000_df["housing_units_total"], errors="coerce")
census_2000_df["jur_clean"] = census_2000_df["NAME"].str.replace(r" city, California$", "", regex=True).map(normalize_jurisdiction)

sd_census_2000 = census_2000_df[census_2000_df["jur_clean"].isin(sd_jur_keys)].copy()
print("SD rows (2000):", len(sd_census_2000))
sd_census_2000.head()


In [ ]:
census_2010_url = "https://api.census.gov/data/2010/dec/sf1"
params_2010 = {
    "get": "NAME,H001001",
    "for": "place:*",
    "in": f"state:{CALIFORNIA_STATE_FIPS}",
    "key": CENSUS_API_KEY,
}

census_2010_raw = get_json(census_2010_url, params=params_2010)
census_2010_df = pd.DataFrame(census_2010_raw[1:], columns=census_2010_raw[0])
census_2010_df = census_2010_df.rename(columns={"H001001": "housing_units_total"})
census_2010_df["housing_units_total"] = pd.to_numeric(census_2010_df["housing_units_total"], errors="coerce")
census_2010_df["jur_clean"] = census_2010_df["NAME"].str.replace(r" city, California$", "", regex=True).map(normalize_jurisdiction)

sd_census_2010 = census_2010_df[census_2010_df["jur_clean"].isin(sd_jur_keys)].copy()
print("SD rows (2010):", len(sd_census_2010))
sd_census_2010.head()


In [ ]:
census_2000_output_path = PROCESSED_DIR / "sd_census_2000_by_jurisdiction.csv"
sd_census_2000.to_csv(census_2000_output_path, index=False)
print("Saved:", census_2000_output_path)

census_2010_output_path = PROCESSED_DIR / "sd_census_2010_by_jurisdiction.csv"
sd_census_2010.to_csv(census_2010_output_path, index=False)
print("Saved:", census_2010_output_path)


## Summary

In [ ]:
# City of SD permits is intentionally not included here - it's an
# optional appendix section that runs later, not part of the core pipeline.
loaded = {
    "APR (permits/completions)": sd_apr_target_year if "sd_apr_target_year" in dir() else pd.DataFrame(),
    "RHNA6 (targets)": sd_rhna6 if "sd_rhna6" in dir() else pd.DataFrame(),
    "DOF (population/housing)": sd_dof if "sd_dof" in dir() else pd.DataFrame(),
    "ACS": sd_acs if "sd_acs" in dir() else pd.DataFrame(),
}
for name, df in loaded.items():
    if df.empty:
        print(f"{name:30s} not loaded")
    else:
        n_missing = (df.isna().sum() > 0).sum()
        print(f"{name:30s} {df.shape[0]} rows, {df.shape[1]} cols, {n_missing} cols with missing values")


## Housing production categories & housing stock fields

In [ ]:
production_categories = pd.DataFrame([
    {"category": "Application submitted", "income_tier": "Acutely Low through Above Moderate (6 tiers)", "source": "APR Table A", "field": "ACUTELY_LOW_INCOME_DR/_NDR ... ABOVE_MOD_INCOME (same 11-column pattern as Table A2's entitlement section)"},
    {"category": "Entitlement", "income_tier": "all tiers", "source": "APR Table A2", "field": "ENT_APPROVE_DT1, NO_ENTITLEMENTS"},
    {"category": "Building permit", "income_tier": "Acutely Low", "source": "APR Table A2", "field": "BP_ACUTELY_LOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Extremely Low", "source": "APR Table A2", "field": "BP_EXTREMELY_LOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Very Low", "source": "APR Table A2", "field": "BP_VLOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Low", "source": "APR Table A2", "field": "BP_LOW_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Moderate", "source": "APR Table A2", "field": "BP_MOD_INCOME_DR / _NDR"},
    {"category": "Building permit", "income_tier": "Above Moderate", "source": "APR Table A2", "field": "BP_ABOVE_MOD_INCOME"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Acutely Low", "source": "APR Table A2", "field": "CO_ACUTELY_LOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Extremely Low", "source": "APR Table A2", "field": "CO_EXTREMELY_LOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Very Low", "source": "APR Table A2", "field": "CO_VLOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Low", "source": "APR Table A2", "field": "CO_LOW_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Moderate", "source": "APR Table A2", "field": "CO_MOD_INCOME_DR / _NDR"},
    {"category": "Completion (certificate of occupancy)", "income_tier": "Above Moderate", "source": "APR Table A2", "field": "CO_ABOVE_MOD_INCOME"},
    {"category": "RHNA target", "income_tier": "VLI / LI / Moderate / Above Moderate", "source": "RHNA6 progress", "field": "RHNA VLI, RHNA LI, RHNA MOD, RHNA ABOVE MOD"},
    {"category": "RHNA progress", "income_tier": "VLI / LI / Moderate / Above Moderate", "source": "RHNA6 progress", "field": "VLI UNITS, LI UNITS, MOD UNITS, ABOVE MOD UNITS"},
    {"category": "City permit approval", "income_tier": "Extremely Low / Very Low / Low / Moderate / Above Moderate", "source": "City of SD permits", "field": "APPROVAL_DU_EXTREMELY_LOW ... APPROVAL_DU_ABOVE_MODERATE"},
    {"category": "ADU", "income_tier": "n/a", "source": "City of SD permits", "field": "APPROVAL_ADU_TOTAL (plus per-tier APPROVAL_ADU_* columns)"},
    {"category": "JADU", "income_tier": "n/a", "source": "City of SD permits", "field": "APPROVAL_JADU_TOTAL (plus per-tier APPROVAL_JADU_* columns)"},
    {"category": "Preservation (existing affordable units retained)", "income_tier": "n/a", "source": "APR Table F (not yet loaded in this notebook)", "field": "see sd_apr_f_preservation_city_year.csv in dashboard prototype"},
])
production_categories.to_csv(DOCS_DIR / "housing_production_categories.csv", index=False)
production_categories


In [ ]:
stock_fields = pd.DataFrame([
    {"field": "Total population", "source": "DOF E-5", "column": "population_total (raw: \"Total\", disambiguated from housing_units_total)"},
    {"field": "Household population", "source": "DOF E-5", "column": "Household"},
    {"field": "Group quarters population", "source": "DOF E-5", "column": "Group Quarters"},
    {"field": "Total housing units", "source": "DOF E-5", "column": "housing_units_total (raw: \"Total.1\")"},
    {"field": "Occupied units", "source": "DOF E-5", "column": "occupied_units (raw: \"Occupied\")"},
    {"field": "Vacant units", "source": "DOF E-5", "column": "vacant_units (computed: housing_units_total - Occupied)"},
    {"field": "Single-family units", "source": "DOF E-5", "column": "single_family_units (computed: Single Detached + Single Attached)"},
    {"field": "Multifamily units", "source": "DOF E-5", "column": "multifamily_units (computed: Two to Four + Five Plus)"},
    {"field": "Mobile home units", "source": "DOF E-5", "column": "mobile_home_units (raw: Mobile Homes)"},
    {"field": "Single detached units (detail)", "source": "DOF E-5", "column": "Single Detached"},
    {"field": "Single attached units (detail)", "source": "DOF E-5", "column": "Single Attached"},
    {"field": "2-4 unit buildings (detail)", "source": "DOF E-5", "column": "Two to Four"},
    {"field": "5+ unit buildings (detail)", "source": "DOF E-5", "column": "Five Plus"},
    {"field": "Vacancy rate", "source": "DOF E-5", "column": "Vacancy Rate"},
    {"field": "Persons per household", "source": "DOF E-5", "column": "Persons per Household"},
    {"field": "Total population", "source": "ACS 5-year", "column": "B01003_001E (population_total)"},
    {"field": "Total housing units", "source": "ACS 5-year", "column": "B25001_001E (housing_units_total)"},
    {"field": "Owner-occupied units", "source": "ACS 5-year", "column": "B25003_002E (owner_occupied)"},
    {"field": "Renter-occupied units", "source": "ACS 5-year", "column": "B25003_003E (renter_occupied)"},
])
stock_fields.to_csv(DOCS_DIR / "housing_stock_fields.csv", index=False)
stock_fields


**Notes on coverage:**

- DOF and ACS both report total population/housing units, but DOF breaks
  housing units out by structure type (single/multi/mobile) while ACS
  breaks out tenure (owner/renter) instead - they're complementary, not
  duplicates, unlike the APR/City-permits overlap found below.
- The APR field names use `VLOW` for "Very Low" while RHNA6 uses `VLI` -
  same tier, different abbreviation between the two datasets.
- DR/NDR suffixes on APR income columns = Deed Restricted / Non-Deed
  Restricted (whether the affordability requirement is legally recorded
  on the property).
- Preservation (Table F) is not yet loaded into this notebook - it exists
  as a processed file in the dashboard prototype repo but isn't part of
  this workstream's live pulls yet.


## Jurisdiction-year RHNA and housing production dataset

In [ ]:
# RHNA6 has no year column (cumulative cycle-to-date, not annual), so its
# target/progress numbers get joined onto each jurisdiction as static
# context rather than matched by year.
#
# Kept broken out by income category (not just totals).
RHNA_TIERS = {
    "very_low": ("RHNA VLI", "VLI UNITS"),
    "low": ("RHNA LI", "LI UNITS"),
    "moderate": ("RHNA MOD", "MOD UNITS"),
    "above_moderate": ("RHNA ABOVE MOD", "ABOVE MOD UNITS"),
}

rhna_summary = sd_rhna6.copy()
tier_cols = ["jur_clean"]

for tier, (target_col, reported_col) in RHNA_TIERS.items():
    rhna_summary[f"rhna_target_{tier}"] = rhna_summary[target_col]
    rhna_summary[f"rhna_reported_{tier}"] = rhna_summary[reported_col]
    # Clipped at 0 - overachieving one tier doesn't offset a shortfall in
    # another; each income tier is a separate obligation, not a shared pool.
    rhna_summary[f"rhna_remaining_{tier}"] = (
        rhna_summary[f"rhna_target_{tier}"] - rhna_summary[f"rhna_reported_{tier}"]
    ).clip(lower=0)
    rhna_summary[f"rhna_pct_achieved_{tier}"] = rhna_summary[f"rhna_reported_{tier}"] / rhna_summary[f"rhna_target_{tier}"]
    tier_cols += [
        f"rhna_target_{tier}", f"rhna_reported_{tier}",
        f"rhna_remaining_{tier}", f"rhna_pct_achieved_{tier}",
    ]

rhna_summary["rhna_target_total"] = rhna_summary[["RHNA VLI", "RHNA LI", "RHNA MOD", "RHNA ABOVE MOD"]].sum(axis=1)
rhna_summary["rhna_units_reported_total"] = rhna_summary[["VLI UNITS", "LI UNITS", "MOD UNITS", "ABOVE MOD UNITS"]].sum(axis=1)
rhna_summary["rhna_remaining_total"] = rhna_summary[[f"rhna_remaining_{t}" for t in RHNA_TIERS]].sum(axis=1)
rhna_summary["rhna_pct_achieved"] = rhna_summary["rhna_units_reported_total"] / rhna_summary["rhna_target_total"]
tier_cols += ["rhna_target_total", "rhna_units_reported_total", "rhna_remaining_total", "rhna_pct_achieved"]

rhna_summary = rhna_summary[tier_cols]

dof_summary = sd_dof.rename(columns={"Household": "population_household"})[[
    "jur_clean", "population_total", "population_household",
    "housing_units_total", "Occupied", "vacant_units",
    "single_family_units", "multifamily_units", "mobile_home_units",
]].rename(columns={"Occupied": "occupied_units"})

jurisdiction_year_dataset = (
    production_by_year
    .merge(applications_by_jurisdiction, on="jur_clean", how="left")
    .merge(rhna_summary, on="jur_clean", how="left")
    .merge(dof_summary, on="jur_clean", how="left")
)

print(jurisdiction_year_dataset.shape)
jurisdiction_year_dataset


In [ ]:
missing = sd_jur_keys - set(jurisdiction_year_dataset["jur_clean"].unique())
if missing:
    print("Jurisdictions missing from the combined table:", sorted(missing))
else:
    print("All 18 cities + County present.")

null_counts = jurisdiction_year_dataset.isna().sum()
null_counts[null_counts > 0]


## San Diego regional total

Separate object, not a 20th row in `jurisdiction_year_dataset` - keeps
countywide and jurisdiction-level results distinct. Sums all 18 cities
plus the unincorporated county.

In [ ]:
assert len(jurisdiction_year_dataset) == 19, (
    f"Expected 18 cities + unincorporated county = 19 rows, got {len(jurisdiction_year_dataset)}"
)

RHNA_TIER_NAMES = ["very_low", "low", "moderate", "above_moderate"]
PRODUCTION_TIER_NAMES = ["very_low", "low", "moderate", "above_moderate"]

SUM_COLS = [
    "application_units_total", "ent_units_total", "bp_units_total", "co_units_total",
    "application_affordable_total", "ent_affordable_total", "bp_affordable_total", "co_affordable_total",
    "project_rows", "rhna_target_total", "rhna_units_reported_total", "rhna_remaining_total",
    "population_total", "population_household",
    "housing_units_total", "occupied_units", "vacant_units",
    "single_family_units", "multifamily_units", "mobile_home_units",
]
SUM_COLS += [f"rhna_target_{t}" for t in RHNA_TIER_NAMES]
SUM_COLS += [f"rhna_reported_{t}" for t in RHNA_TIER_NAMES]
SUM_COLS += [f"rhna_remaining_{t}" for t in RHNA_TIER_NAMES]
SUM_COLS += [f"bp_{t}_total" for t in PRODUCTION_TIER_NAMES]
SUM_COLS += [f"co_{t}_total" for t in PRODUCTION_TIER_NAMES]

sd_region_total = jurisdiction_year_dataset[SUM_COLS].sum().to_frame().T
sd_region_total.insert(0, "jurisdiction", "San Diego Region (18 cities + Unincorporated County)")
sd_region_total.insert(1, "year", TARGET_YEAR)
sd_region_total.insert(2, "jurisdictions_included", len(jurisdiction_year_dataset))

# Share/percentage fields recalculated at the regional level, not summed
# or averaged directly.
sd_region_total["application_affordable_share"] = sd_region_total["application_affordable_total"] / sd_region_total["application_units_total"]
sd_region_total["ent_affordable_share"] = sd_region_total["ent_affordable_total"] / sd_region_total["ent_units_total"]
sd_region_total["bp_affordable_share"] = sd_region_total["bp_affordable_total"] / sd_region_total["bp_units_total"]
sd_region_total["co_affordable_share"] = sd_region_total["co_affordable_total"] / sd_region_total["co_units_total"]
sd_region_total["rhna_pct_achieved"] = sd_region_total["rhna_units_reported_total"] / sd_region_total["rhna_target_total"]

for tier in RHNA_TIER_NAMES:
    sd_region_total[f"rhna_pct_achieved_{tier}"] = (
        sd_region_total[f"rhna_reported_{tier}"] / sd_region_total[f"rhna_target_{tier}"]
    )

sd_region_total


In [ ]:
check = jurisdiction_year_dataset["bp_units_total"].sum() == sd_region_total["bp_units_total"].iloc[0]
print("Regional bp_units_total matches sum of jurisdiction rows:", check)

region_output_path = PROCESSED_DIR / f"rhna_housing_production_{TARGET_YEAR}_regional_total.csv"
sd_region_total.to_csv(region_output_path, index=False)
print("Saved:", region_output_path)


## Power BI-ready export (unified long format)

One fact table combining RHNA, all 4 production stages, production by
housing type, housing stock, per-1,000-resident ratios, and all 4
historical benchmarks - schema: `jurisdiction, year, metric,
income_category, value, source, unit_of_measure`. Everything Power BI
needs to filter and build cards/tables from a single import, instead of
juggling several files with different shapes.

In [ ]:
TIER_LABELS = {"very_low": "Very Low", "low": "Low", "moderate": "Moderate", "above_moderate": "Above Moderate"}
unified_rows = []

def add_rows(df, jur_col, year_col_or_value, metric, tier_col_map, total_col, source, unit):
    """tier_col_map: {tier_key: column_name}. total_col: column name for the Total row."""
    for tier_key, col in tier_col_map.items():
        if col not in df.columns:
            continue
        chunk = df[[jur_col]].copy()
        chunk["year"] = df[year_col_or_value] if isinstance(year_col_or_value, str) else year_col_or_value
        chunk["metric"] = metric
        chunk["income_category"] = TIER_LABELS[tier_key]
        chunk["value"] = df[col]
        chunk["source"] = source
        chunk["unit_of_measure"] = unit
        unified_rows.append(chunk.rename(columns={jur_col: "jurisdiction"}))
    if total_col and total_col in df.columns:
        chunk = df[[jur_col]].copy()
        chunk["year"] = df[year_col_or_value] if isinstance(year_col_or_value, str) else year_col_or_value
        chunk["metric"] = metric
        chunk["income_category"] = "Total"
        chunk["value"] = df[total_col]
        chunk["source"] = source
        chunk["unit_of_measure"] = unit
        unified_rows.append(chunk.rename(columns={jur_col: "jurisdiction"}))


def add_flat_row(df, jur_col, year_value, metric, value_col, source, unit):
    chunk = df[[jur_col]].copy()
    chunk["year"] = year_value
    chunk["metric"] = metric
    chunk["income_category"] = "Total"
    chunk["value"] = df[value_col]
    chunk["source"] = source
    chunk["unit_of_measure"] = unit
    unified_rows.append(chunk.rename(columns={jur_col: "jurisdiction"}))


RHNA_SOURCE = "HCD RHNA 6th Cycle Progress Report"
add_rows(jurisdiction_year_dataset, "jur_clean", TARGET_YEAR, "RHNA Allocation",
         {t: f"rhna_target_{t}" for t in TIER_LABELS}, "rhna_target_total", RHNA_SOURCE, "Housing units (count)")
add_rows(jurisdiction_year_dataset, "jur_clean", TARGET_YEAR, "RHNA Qualifying Units",
         {t: f"rhna_reported_{t}" for t in TIER_LABELS}, "rhna_units_reported_total", RHNA_SOURCE, "Housing units (count)")
add_rows(jurisdiction_year_dataset, "jur_clean", TARGET_YEAR, "RHNA Remaining",
         {t: f"rhna_remaining_{t}" for t in TIER_LABELS}, "rhna_remaining_total", RHNA_SOURCE, "Housing units (count)")
add_rows(jurisdiction_year_dataset, "jur_clean", TARGET_YEAR, "RHNA Percent Complete",
         {t: f"rhna_pct_achieved_{t}" for t in TIER_LABELS}, "rhna_pct_achieved", RHNA_SOURCE, "Percent (0-1 share)")

print(f"RHNA rows added: {sum(len(r) for r in unified_rows)}")


In [ ]:
STAGE_SOURCES = {
    "Applications": (applications_history, "application", "HCD APR Table A"),
    "Entitlements": (production_history, "ent", "HCD APR Table A2"),
    "Permits": (production_history, "bp", "HCD APR Table A2"),
    "Completed": (production_history, "co", "HCD APR Table A2"),
}
for metric, (df, prefix, source) in STAGE_SOURCES.items():
    add_rows(df, "jur_clean", "year", metric,
             {t: f"{prefix}_{t}_total" for t in TIER_LABELS}, f"{prefix}_units_total", source, "Housing units (count)")

print(f"Running total after production stages: {sum(len(r) for r in unified_rows)}")


In [ ]:
TYPE_SOURCE = "HCD APR Table A2 (UNIT_CAT)"

for _, row in production_by_type.iterrows():
    for stage_label, col in [("Permitted", "bp_units_total"), ("Completed", "co_units_total")]:
        unified_rows.append(pd.DataFrame([{
            "jurisdiction": row["jur_clean"],
            "year": TARGET_YEAR,
            "metric": f"Production - {row['housing_type']} ({stage_label})",
            "income_category": "Total",
            "value": row[col],
            "source": TYPE_SOURCE,
            "unit_of_measure": "Housing units (count)",
        }]))

print(f"Running total after production by type: {sum(len(r) for r in unified_rows)}")


In [ ]:
DOF_SOURCE = "CA DOF E-5"

add_flat_row(jurisdiction_year_dataset, "jur_clean", TARGET_YEAR, "Total Housing Stock", "housing_units_total", DOF_SOURCE, "Housing units (count)")
add_flat_row(jurisdiction_year_dataset, "jur_clean", TARGET_YEAR, "Occupied Housing Units", "occupied_units", DOF_SOURCE, "Housing units (count)")
add_flat_row(jurisdiction_year_dataset, "jur_clean", TARGET_YEAR, "Vacant Housing Units", "vacant_units", DOF_SOURCE, "Housing units (count)")
add_flat_row(jurisdiction_year_dataset, "jur_clean", TARGET_YEAR, "Single-Family Housing Stock", "single_family_units", DOF_SOURCE, "Housing units (count)")
add_flat_row(jurisdiction_year_dataset, "jur_clean", TARGET_YEAR, "Multifamily Housing Stock", "multifamily_units", DOF_SOURCE, "Housing units (count)")
add_flat_row(jurisdiction_year_dataset, "jur_clean", TARGET_YEAR, "Mobile Home Housing Stock", "mobile_home_units", DOF_SOURCE, "Housing units (count)")
add_flat_row(jurisdiction_year_dataset, "jur_clean", TARGET_YEAR, "Annual DOF Housing Stock", "housing_units_total", DOF_SOURCE, "Housing units (count)")

# Computed on a copy, not in place -- jurisdiction_year_dataset gets
# exported as its own CSV later (## Export) and should not silently gain
# extra columns as a side effect of building this Power BI table.
ratios = jurisdiction_year_dataset[["jur_clean", "bp_units_total", "housing_units_total", "population_total"]].copy()
ratios["permits_per_1000"] = ratios["bp_units_total"] / ratios["population_total"] * 1000
ratios["housing_units_per_1000"] = ratios["housing_units_total"] / ratios["population_total"] * 1000
add_flat_row(ratios, "jur_clean", TARGET_YEAR, "Permits per 1,000 Residents", "permits_per_1000", DOF_SOURCE, "Units per 1,000 residents")
add_flat_row(ratios, "jur_clean", TARGET_YEAR, "Housing Units per 1,000 Residents", "housing_units_per_1000", DOF_SOURCE, "Units per 1,000 residents")

print(f"Running total after stock/ratios: {sum(len(r) for r in unified_rows)}")


In [ ]:
# Historical benchmarks - each year tagged with its actual source and
# its real collection year, not just the 4-digit label.
add_flat_row(sd_census_2000, "jur_clean", 2000, "Housing Stock Benchmark", "housing_units_total",
             "US Census 2000 Decennial (SF1, point-in-time)", "Housing units (count)")
add_flat_row(sd_census_2010, "jur_clean", 2010, "Housing Stock Benchmark", "housing_units_total",
             "US Census 2010 Decennial (SF1, point-in-time)", "Housing units (count)")
add_flat_row(sd_acs_2021, "jur_clean", 2021, "Housing Stock Benchmark", "housing_units_total",
             "US Census ACS 2017-2021 5-year (rolling average)", "Housing units (count)")
add_flat_row(sd_acs, "jur_clean", ACS_DATA_YEAR, "Housing Stock Benchmark", "housing_units_total",
             f"US Census ACS {ACS_VINTAGE_LABEL} 5-year (rolling average)", "Housing units (count)")

powerbi_unified = pd.concat(unified_rows, ignore_index=True)
print(powerbi_unified.shape)
powerbi_unified.head(10)


### Self-test (reproduces the Power BI validation card)

Confirms the exported file will actually show the right number before
anyone opens Power BI - same check as the guide's Part 8 card
(San Diego, RHNA Allocation, Total, 2025 -> expect 108036).

In [ ]:
test_value = powerbi_unified[
    (powerbi_unified["jurisdiction"] == "san diego")
    & (powerbi_unified["metric"] == "RHNA Allocation")
    & (powerbi_unified["income_category"] == "Total")
    & (powerbi_unified["year"] == 2025)
]["value"].iloc[0]

print("San Diego RHNA Allocation, Total, 2025:", test_value)
assert test_value == 108036, f"Expected 108036, got {test_value}"
print("Self-test passed.")

powerbi_path = PROCESSED_DIR / f"powerbi_rhna_production_{APR_START_YEAR}_{TARGET_YEAR}_long.csv"
powerbi_unified.to_csv(powerbi_path, index=False)
print("Saved:", powerbi_path)


## Metric dictionary

In [ ]:
metric_dictionary = pd.DataFrame([
    {
        "output_metric": "application_units_total",
        "source_table": "HCD APR Table A",
        "source_variables": "Sum of unprefixed *_INCOME_* columns, all APPLICATION_STATUS values included",
        "definition": "Total housing units in applications submitted, all income tiers, per jurisdiction-year. Counts all applications regardless of approval status - this is 'submitted', not 'approved'. Cross-checked against Table A's own TOT_PROPOSED_UNITS field",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Small gaps vs. TOT_PROPOSED_UNITS for 5 of 19 jurisdictions - see data_quality_limitations.csv",
    },
    {
        "output_metric": "ent_units_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of unprefixed *_INCOME_* columns (not BP_ or CO_ prefixed)",
        "definition": "Total housing units with a planning entitlement approved, all income tiers, per jurisdiction-year. Kept fully separate from bp_units_total and co_units_total - never summed together",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Matches HCD's own auto-populated NO_ENTITLEMENTS total exactly for all 19 jurisdictions",
    },
    {
        "output_metric": "bp_units_total / bp_<tier>_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of BP_*_INCOME columns, kept both as an overall total and broken out per income tier (very_low, low, moderate, above_moderate)",
        "definition": "Total housing units with a building permit issued, per jurisdiction-year. Per-tier columns sum back to the overall total exactly (enforced by an assertion in the notebook)",
        "source_year": f"2018-{TARGET_YEAR} (historical export); {TARGET_YEAR} (jurisdiction-year table)",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Self-reported by jurisdictions to HCD, not independently verified",
    },
    {
        "output_metric": "co_units_total / co_<tier>_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of CO_*_INCOME columns, kept both as an overall total and broken out per income tier (very_low, low, moderate, above_moderate)",
        "definition": "Total housing units with a certificate of occupancy (completed), per jurisdiction-year. Per-tier columns sum back to the overall total exactly (enforced by an assertion in the notebook)",
        "source_year": f"2018-{TARGET_YEAR} (historical export); {TARGET_YEAR} (jurisdiction-year table)",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Self-reported by jurisdictions to HCD, not independently verified",
    },
    {
        "output_metric": "bp_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "bp_affordable_total / bp_units_total",
        "definition": "Share of permitted units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Percent (0-1 share)",
        "limitations": "NaN when bp_units_total is 0 that year - zero permits, not missing data",
    },
    {
        "output_metric": "co_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "co_affordable_total / co_units_total",
        "definition": "Share of completed units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county)",
        "unit_of_measurement": "Percent (0-1 share)",
        "limitations": "NaN when co_units_total is 0 that year - zero completions, not missing data",
    },
    {
        "output_metric": "rhna_target_<tier>",
        "source_table": "HCD RHNA 6th Cycle Progress Report (Table B)",
        "source_variables": "RHNA VLI, RHNA LI, RHNA MOD, RHNA ABOVE MOD",
        "definition": "Assigned RHNA target units, kept separate per income tier (very_low, low, moderate, above_moderate) for the 6th Cycle planning period, per jurisdiction - plus rhna_target_total for the summed value",
        "source_year": "6th Cycle (2021-2029), adopted allocation, does not change during the cycle",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county) and regional total",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Validated exact match against HCD/SANDAG published figures",
    },
    {
        "output_metric": "rhna_reported_<tier> / rhna_pct_achieved_<tier> / rhna_remaining_<tier>",
        "source_table": "HCD RHNA 6th Cycle Progress Report (Table B)",
        "source_variables": "VLI UNITS, LI UNITS, MOD UNITS, ABOVE MOD UNITS",
        "definition": "BASED ON BUILDING PERMITS ISSUED, not completed units - this is HCD's own RHNA-credit methodology. Kept separate per income tier, per jurisdiction. Cumulative cycle-to-date, not year-by-year. Do not conflate with co_units_total (completions)",
        "source_year": f"Cumulative, 6th Cycle to date (as of {TARGET_YEAR} data pull)",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated county) and regional total",
        "unit_of_measurement": "Housing units (count); Percent for rhna_pct_achieved_<tier>",
        "limitations": "Cumulative cycle-to-date, not annual",
    },
    {
        "output_metric": "sd_permit_du_by_tier",
        "source_table": "City of San Diego Development Permits (Active + Closed approvals)",
        "source_variables": "APPROVAL_DU_EXTREMELY_LOW/VERY_LOW/LOW/MODERATE/ABOVE_MODERATE",
        "definition": "Dwelling units per approval, broken out by income tier, City of San Diego only. Separate ADU (APPROVAL_ADU_*) and JADU (APPROVAL_JADU_*) columns exist alongside standard units",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "City of San Diego only",
        "unit_of_measurement": "Housing units (count)",
        "limitations": "Overlaps with APR for San Diego - do not sum with bp_units_total. Appendix data, not in core exports",
    },
    {
        "output_metric": "sd_permit_stage",
        "source_table": "City of San Diego Development Permits (Active + Closed approvals)",
        "source_variables": "APPROVAL_ISSUE_DATE, APPROVAL_CLOSE_DATE, approval_status",
        "definition": "This dataset tracks permit issuance and closure only - it has no certificate-of-occupancy / completion field equivalent to APR's CO_* columns",
        "source_year": TARGET_YEAR, "download_date": DOWNLOAD_DATE,
        "geographic_level": "City of San Diego only",
        "unit_of_measurement": "Date / status (not a count)",
        "limitations": "'Closed' can mean finaled, expired, or withdrawn - not necessarily completed construction",
    },
])
metric_dictionary.to_csv(DOCS_DIR / "rhna_housing_production_metric_dictionary.csv", index=False)
metric_dictionary


## Export

In [ ]:
output_path = PROCESSED_DIR / f"rhna_housing_production_{TARGET_YEAR}_by_jurisdiction.csv"
jurisdiction_year_dataset.to_csv(output_path, index=False)
print("Saved:", output_path)


### Export metadata

Dataset-level metadata for each exported file - kept as its own record
rather than repeated on every row.

In [ ]:
export_metadata = pd.DataFrame([
    {
        "file_name": f"rhna_housing_production_{TARGET_YEAR}_by_jurisdiction.csv",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated San Diego County)",
        "unit_of_measurement": "Housing units (count); percent for share/pct_achieved fields",
        "limitations": "See data_quality_limitations.csv for full detail",
    },
    {
        "file_name": f"apr_production_{APR_START_YEAR}_{TARGET_YEAR}_by_jurisdiction_year.csv",
        "source_year": f"{APR_START_YEAR}-{TARGET_YEAR}",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated San Diego County), by year",
        "unit_of_measurement": "Housing units (count); percent for affordable-share fields",
        "limitations": "2018 (APR's first collection year) may have lower reporting completeness than later years",
    },
    {
        "file_name": f"rhna_housing_production_{TARGET_YEAR}_regional_total.csv",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Regional (all 18 cities + unincorporated San Diego County combined)",
        "unit_of_measurement": "Housing units (count); percent for share/pct_achieved fields",
        "limitations": "Percent fields recalculated at the regional level, not averaged from jurisdiction percentages",
    },
    {
        "file_name": f"powerbi_rhna_production_{APR_START_YEAR}_{TARGET_YEAR}_long.csv",
        "source_year": f"2000-{TARGET_YEAR} (varies by metric - see the metric/source columns within the file itself)",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated San Diego County, except ACS/Census-based rows which are 18 cities only)",
        "unit_of_measurement": "Varies by metric - see the file's own unit_of_measure column",
        "limitations": "RHNA, stock, and housing-type rows are single-year (2025) snapshots; only the 4 production stages (Applications/Entitlements/Permits/Completed) span the full 2018-2025 range. Check per-year jurisdiction coverage before filtering",
    },
    {
        "file_name": f"apr_production_by_housing_type_{TARGET_YEAR}.csv",
        "source_year": TARGET_YEAR,
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities + unincorporated San Diego County)",
        "unit_of_measurement": "Housing units (count), broken out by structure type (UNIT_CAT), not income tier",
        "limitations": "UNIT_CAT category mapping needs confirmation against real value_counts() output - see the TODO in the Production by housing type section",
    },
    {
        "file_name": f"sd_acs_{ACS_DATA_YEAR}_by_jurisdiction.csv",
        "source_year": f"{ACS_VINTAGE_LABEL} (represents {ACS_DATA_YEAR})",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities only - ACS places do not cover unincorporated county)",
        "unit_of_measurement": "Population, housing units (count); owner/renter occupied (count)",
        "limitations": "5-year rolling estimate with margins of error, largest for small jurisdictions",
    },
    {
        "file_name": f"sd_acs_{ACS_2021_DATA_YEAR}_by_jurisdiction.csv",
        "source_year": f"{ACS_2021_VINTAGE_LABEL} (represents {ACS_2021_DATA_YEAR})",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (18 cities only - ACS places do not cover unincorporated county)",
        "unit_of_measurement": "Population, housing units (count); owner/renter occupied (count)",
        "limitations": "5-year rolling estimate (2017-2021), not directly comparable to Decennial Census point-in-time counts",
    },
    {
        "file_name": "sd_census_2000_by_jurisdiction.csv",
        "source_year": "2000 (April 1, point-in-time count)",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (places only - 2000 Census place boundaries may not exactly match current jurisdiction boundaries)",
        "unit_of_measurement": "Total housing units (count) - Summary File 1, variable H001001",
        "limitations": "Full-count Census, not a survey estimate. Place-boundary changes over 25 years may affect comparability",
    },
    {
        "file_name": "sd_census_2010_by_jurisdiction.csv",
        "source_year": "2010 (April 1, point-in-time count)",
        "download_date": DOWNLOAD_DATE,
        "geographic_level": "Jurisdiction (places only - 2010 Census place boundaries may not exactly match current jurisdiction boundaries)",
        "unit_of_measurement": "Total housing units (count) - Summary File 1, variable H001001",
        "limitations": "Full-count Census, not a survey estimate. Place-boundary changes over 15 years may affect comparability",
    },
])
export_metadata.to_csv(DOCS_DIR / "export_metadata.csv", index=False)
export_metadata


## Validate against HCD's published RHNA results

Checks `rhna_target_total` against figures from official adopted planning
documents (SANDAG Board Resolution, adopted Housing Elements) - a
different publication channel than the CKAN dataset this notebook pulls
from.

In [ ]:
PUBLISHED_RHNA = pd.DataFrame([
    {"jur_clean": "san diego", "published_rhna_target": 108036,
     "source": "City of San Diego adopted Housing Element 2021-2029"},
    {"jur_clean": "unincorporated san diego county", "published_rhna_target": 6700,
     "source": "County of San Diego General Plan Annual Progress Report"},
    {"jur_clean": "oceanside", "published_rhna_target": 5443,
     "source": "City of Oceanside Housing Element page"},
])

rhna_check = jurisdiction_year_dataset[["jur_clean", "rhna_target_total"]].merge(
    PUBLISHED_RHNA, on="jur_clean", how="inner"
)
rhna_check["diff"] = rhna_check["rhna_target_total"] - rhna_check["published_rhna_target"]
print("Jurisdiction-level check:")
display(rhna_check)

regional_published = 171685
regional_ours = sd_region_total["rhna_target_total"].iloc[0]
print(f"\nRegional total: ours={regional_ours:.0f}, published={regional_published}, "
      f"match={regional_ours == regional_published}")


**Result:** exact match, regional total + all 3 jurisdictions checked.
Validates `rhna_target_total` only - `rhna_reported_*` is cross-checked
separately against `NO_ENTITLEMENTS`/etc. in the APR section (HCD doesn't
publish a canonical "progress" number the way it does for targets).

## Validation against dashboard prototype

In [ ]:
def find_sibling_repo(repo_name: str, search_depth: int = 3) -> Path | None:
    candidates = [ROOT, *ROOT.parents][:search_depth + 2]
    for base in candidates:
        match = base / repo_name
        if match.exists():
            return match
        if base.parent.exists():
            for sibling in base.parent.iterdir():
                if sibling.name == repo_name and sibling.is_dir():
                    return sibling
    return None


baseline_repo = find_sibling_repo("housing-dashboard-prototype")

if baseline_repo is None:
    print(
        "Could not find a local clone of housing-dashboard-prototype near this repo. "
        "Clone it into the same parent folder as this repo, then re-run this cell."
    )
else:
    BASELINE_PATH = baseline_repo / "data" / "processed" / "sd_apr_a2_city_year_supply.csv"
    print("Found baseline repo at:", baseline_repo)


In [ ]:
if baseline_repo is not None and BASELINE_PATH.exists():
    baseline = pd.read_csv(BASELINE_PATH)
    baseline["jur_clean"] = baseline["jur_clean"].str.lower()

    baseline_years = sorted(baseline["year"].unique())
    our_years = sorted(production_history["year"].dropna().unique())
    overlapping_years = sorted(set(baseline_years) & set(our_years))
    years_ahead = sorted(set(our_years) - set(baseline_years))

    if years_ahead:
        print(
            f"Baseline (dashboard prototype) does not yet include: {years_ahead}. "
            "This notebook's pull is ahead of the dashboard prototype for those years - "
            "excluded from the comparison below, not treated as mismatches."
        )

    if not overlapping_years:
        print("No overlapping years between this notebook's pull and the baseline - nothing to validate.")
    else:
        comparison = production_history.merge(
            baseline, on=["jur_clean", "year"], suffixes=("_fresh", "_baseline"), how="inner",
        )
        failed_checks = comparison[
            comparison["bp_units_total_fresh"] != comparison["bp_units_total_baseline"]
        ]
        print(f"Validated years: {overlapping_years}")
        print(f"Compared {len(comparison)} rows; {len(failed_checks)} mismatches.")
        display(failed_checks[["jur_clean", "year", "bp_units_total_fresh", "bp_units_total_baseline"]])
        comparison.to_csv(PROCESSED_DIR / "rhna_housing_production_validation_report.csv", index=False)
elif baseline_repo is not None:
    print(f"Repo found but expected file is missing: {BASELINE_PATH}")


## Appendix: City of San Diego permits (optional)

Not part of the core pipeline - doesn't feed jurisdiction_year_dataset or
any export. Used only to validate APR's San Diego permit numbers against
the City's own system (overlap check below).

In [ ]:
permits_dir = RAW_DIR.parent / "sandiego_permits"
permits_dir.mkdir(parents=True, exist_ok=True)

permits_frames = []
for label in ["active", "closed"]:
    raw_path = permits_dir / f"{label}_approvals_raw.csv"
    if not raw_path.exists():
        print(f"Missing: {raw_path}")
        print("Download from https://data.sandiego.gov/datasets/development-permits-set2/ and save it there.")
        continue
    df = pd.read_csv(raw_path, low_memory=False)
    df["approval_status"] = label
    permits_frames.append(df)

if permits_frames:
    sd_permits_raw = pd.concat(permits_frames, ignore_index=True)
    print(sd_permits_raw.shape)
else:
    sd_permits_raw = pd.DataFrame()
sd_permits_raw.head()


In [ ]:
print(sd_permits_raw.columns.tolist())

In [ ]:
sd_permits_raw["APPROVAL_ISSUE_DATE"] = pd.to_datetime(sd_permits_raw["APPROVAL_ISSUE_DATE"], errors="coerce")

DU_TIER_COLS = [
    "APPROVAL_DU_EXTREMELY_LOW", "APPROVAL_DU_VERY_LOW", "APPROVAL_DU_LOW",
    "APPROVAL_DU_MODERATE", "APPROVAL_DU_ABOVE_MODERATE",
]
ADU_JADU_COLS = ["APPROVAL_ADU_TOTAL", "APPROVAL_JADU_TOTAL"]

sd_permits_raw["du_tier_total"] = sd_permits_raw[DU_TIER_COLS].fillna(0).sum(axis=1)
sd_permits_raw["adu_jadu_total"] = sd_permits_raw[ADU_JADU_COLS].fillna(0).sum(axis=1)

has_du_impact = (sd_permits_raw["du_tier_total"] != 0) | (sd_permits_raw["adu_jadu_total"] != 0)
in_target_year = sd_permits_raw["APPROVAL_ISSUE_DATE"].dt.year == TARGET_YEAR

sd_permits_housing = sd_permits_raw[has_du_impact & in_target_year].copy()
print(f"Housing-relevant permits, {TARGET_YEAR}:", len(sd_permits_housing))
print("Of", len(sd_permits_raw), "total raw rows")
sd_permits_housing[["PROJECT_TITLE", "JOB_BC_CODE_DESCRIPTION", "du_tier_total", "adu_jadu_total"]].head(10)


## Check APR vs. City of SD permit overlap

In [ ]:
apr_sd_only = sd_apr_target_year[sd_apr_target_year["jur_clean"] == "san diego"]
apr_sd_units = apr_sd_only["bp_units_row"].sum()

city_permits_total = sd_permits_housing["du_tier_total"].sum() + sd_permits_housing["adu_jadu_total"].sum()

print(f"APR, City of San Diego, {TARGET_YEAR} permitted units: {apr_sd_units:.0f}")
print(f"City of SD permit system, {TARGET_YEAR} units (DU tiers + ADU/JADU): {city_permits_total:.0f}")
print(f"Ratio (city system / APR): {city_permits_total / apr_sd_units:.2f}" if apr_sd_units else "APR total is zero")


In [ ]:
apr_sd_apns = set(apr_sd_only["APN"].dropna().astype(str).str.strip())
city_apns = set(sd_permits_housing["GIS_APN"].dropna().astype(str).str.strip())

overlap_apns = apr_sd_apns & city_apns
print(f"APR SD APNs ({TARGET_YEAR}): {len(apr_sd_apns)}")
print(f"City permit APNs ({TARGET_YEAR}): {len(city_apns)}")
print(f"APNs appearing in both: {len(overlap_apns)}")
print(f"Share of City permit APNs also in APR: {len(overlap_apns) / len(city_apns):.1%}" if city_apns else "no city APNs")


**Finding:** APR and City of SD permits report largely the same
underlying activity (totals close, ~93% APN match) - don't sum them.
Doesn't apply to the other 17 jurisdictions; APR is their only source.

## Missing data, unclear fields, and source limitations

In [ ]:
data_quality_log = pd.DataFrame([
    {"type": "Missing data", "source": "APR Table A2", "item": "bp_affordable_share / co_affordable_share",
     "note": "NaN when bp_units_total or co_units_total is 0 for that jurisdiction-year (0/0), not a data error - means zero permits/completions that year, not unknown affordability"},
    {"type": "Missing data", "source": "APR Table A2", "item": "NOTES, LATITUDE/LONGITUDE, DR_TYPE, FIN_ASSIST_NAME, PRIOR_APN",
     "note": "Conditional fields - only populated for specific project types (e.g. DR_TYPE only for deed-restricted units). Sparse by design, not incomplete"},
    {"type": "Missing data", "source": "City of SD permits", "item": "ADU/JADU and income-tier DU columns",
     "note": "Blank on non-residential permit rows (electrical, plumbing, signage, etc.) - expected, filtered out of sd_permits_housing"},
    {"type": "Missing data", "source": "APR Table F (preservation)", "item": "entire table",
     "note": "Not yet loaded into this workstream. Exists as a processed file in the dashboard prototype repo (sd_apr_f_preservation_city_year.csv) but not pulled live here"},
    {"type": "Missing data", "source": "This workstream", "item": "NOAH (naturally occurring affordable housing) loss estimate",
     "note": "No published dataset exists for this - would need to be derived from ACS rent + building-age data. Not started"},
    {"type": "Missing data", "source": "HCD APR (general)", "item": "current reporting year, some jurisdictions",
     "note": "Jurisdictions can file late; a given year's data may be incomplete for months after the April 1 deadline. Check the 'not found' warning printed when loading before trusting a fresh pull"},
    {"type": "Missing data", "source": "HCD APR", "item": "units under construction",
     "note": "Not tracked anywhere in HCD's APR data - no table captures this milestone. Application, entitlement, permit, and completion stages are all available (Table A and Table A2), but under-construction status is not"},
    {"type": "Missing data", "source": "APR Table A2, City of San Diego", "item": "ent_units_total near-zero relative to bp_units_total",
     "note": "San Diego reported only 3 entitled units in 2025 against 7,842 permitted units - an unusually large gap for the county's largest jurisdiction. Likely a self-reporting gap in San Diego's own APR submission for the entitlement stage specifically, not evidence that entitlement activity actually stopped"},
    {"type": "Unclear field", "source": "APR Table A2", "item": "*_DR / *_NDR column suffixes",
     "note": "Deed Restricted / Non-Deed Restricted (whether the affordability requirement is legally recorded on the property). Not explained in the CSV itself - confirmed from HCD's separate Table A2 data dictionary (.docx)"},
    {"type": "Unclear field", "source": "APR Table A2 vs. RHNA6", "item": "VLOW vs. VLI",
     "note": "Same income tier (Very Low Income), different abbreviation between two HCD datasets. Easy to miss if joining/comparing by tier name"},
    {"type": "Unclear field", "source": "RHNA6 progress report", "item": "what \"units reported\" actually measures",
     "note": "RHNA progress in this dataset is based on BUILDING PERMITS ISSUED, not completed units - confirmed via HCD's own APR guidance (\"only building permits are used for the purposes of determining progress towards RHNA\"). Entitlements and completions are tracked elsewhere in the APR but do not count toward RHNA credit"},
    {"type": "Unclear field", "source": "City of SD permits", "item": "APPROVAL_DU_NET_CHANGE",
     "note": "Misleadingly named - sums to exactly 0 across a full year of data, unreliable. Real unit counts live in the separate income-tier APPROVAL_DU_* columns instead"},
    {"type": "Resolved / corrected", "source": "APR Table A2", "item": "NO_ENTITLEMENTS / NO_BUILDING_PERMITS / NO_OTHER_FORMS_OF_READINESS",
     "note": "CORRECTED: previously mischaracterized as flags. Per HCD's official APR instructions, these mean \"Number Of\" - auto-populated total-unit-count fields calculated by HCD from the same per-tier income columns this notebook sums independently. Cross-check against ent_units_total / bp_units_total / co_units_total now matches exactly for all 19/19 jurisdictions, all three stages"},
    {"type": "Resolved / corrected", "source": "This notebook", "item": "ENT_INCOME_COLS incorrectly included EXTR_LOW_INCOME_UNITS",
     "note": "A loose text search (\"contains INCOME, not BP_/CO_ prefixed\") swept in an unrelated Table A2 field, inflating ent_units_total beyond what the 4 tier columns summed to. Caused an apparent 55-unit Escondido anomaly that was never a real data-source issue - fixed by building the column list explicitly from the known tier structure instead of a text search"},
    {"type": "Missing data", "source": "APR Table A", "item": "application_units_total vs. TOT_PROPOSED_UNITS small discrepancies",
     "note": "14 of 19 jurisdictions match exactly; 5 show small gaps (largest: Escondido, 88 units / ~16% relative). Likely reflects units reported in TOT_PROPOSED_UNITS without a corresponding income-tier breakdown yet, rather than a calculation error"},
    {"type": "Missing data", "source": "APR Table A (historical, 2018-2020)", "item": "one jurisdiction per year with zero applications reported",
     "note": "Lemon Grove (2018), Imperial Beach (2019), and San Marcos (2020) each show zero application rows for that year - filled with 0 in the Power BI export via fill_missing_jurisdiction_years(), but genuinely absent from HCD's raw Table A data for those specific jurisdiction-years"},
    {"type": "Unclear field", "source": "HCD / DOF / dashboard prototype", "item": "County jurisdiction naming",
     "note": "Appears as 'Unincorporated' (DOF), 'SAN DIEGO COUNTY' (APR/RHNA), and 'County of San Diego' in various places - required manual normalization to a single consistent key ('unincorporated san diego county') across this notebook"},
    {"type": "Source limitation", "source": "APR / RHNA (HCD)", "item": "self-reported",
     "note": "Not independently verified by HCD. Quality, completeness, and filing timeliness vary by jurisdiction"},
    {"type": "Source limitation", "source": "RHNA6 progress report", "item": "cumulative only",
     "note": "No year-by-year breakdown - reports cycle-to-date totals only (2021-2029), can't see year-over-year pace toward the target from this file alone"},
    {"type": "Source limitation", "source": "City of SD permits", "item": "single-jurisdiction coverage",
     "note": "Only covers the City of San Diego, not the other 17 jurisdictions. Also overlaps with APR for San Diego specifically (~1.04 ratio, 93% APN match) - don't sum the two for San Diego totals"},
    {"type": "Source limitation", "source": "DOF E-5", "item": "annual point-in-time estimate",
     "note": "January 1 snapshot, not real-time. Uses different methodology than ACS, so the two won't match exactly even for the same year"},
    {"type": "Source limitation", "source": "Census ACS", "item": "one year behind target year",
     "note": "Newest available vintage is 2020-2024 (\"2024\" data) while APR/DOF/permits target 2025 - ACS structurally cannot produce 2025 data until ~Dec 2026/Jan 2027. Also a 5-year rolling estimate with margins of error, largest for small jurisdictions like Del Mar (~3,900 population)"},
    {"type": "Unclear field", "source": "Housing Stock Benchmark rows (2000/2010/2021/2024)", "item": "point-in-time count vs. rolling average", "note": "2000/2010 Census benchmarks (not yet pulled) are true point-in-time counts, taken on a single date. 2021/2024 ACS benchmarks are 5-year rolling averages (2017-2021 and 2020-2024 windows) blended across multiple years of survey responses, not a snapshot of any single year. Comparing 2010 to 2021 is not apples-to-apples the same way 2000 to 2010 would be - one is a snapshot, the other a multi-year blend"},
    {"type": "Source limitation", "source": "Dashboard prototype (housing-dashboard-prototype repo)", "item": "stale baseline for validation",
     "note": "The prototype's sd_apr_a2_city_year_supply.csv only covers 2018-2024 - it has no 2025 data yet, so this notebook's 2025 pull currently has nothing to validate against for that year. Not an error in this notebook; the prototype simply hasn't been refreshed with 2025 APR data"},
    {"type": "Source limitation", "source": "HCD APR (general)", "item": "historical years get amended after publication",
     "note": "Validating 2018-2024 against the dashboard prototype found 9 of 126 city-year rows differ (out of 18 cities x 7 years). In every case this notebook's fresh pull is HIGHER than the prototype's older snapshot, never lower, and gaps are small (3-59 units) - consistent with HCD amending past years' data (late filings, corrections) after initial publication. Largest gap: San Marcos 2024 (455 vs. 396, ~13%)"},
])

data_quality_log.to_csv(DOCS_DIR / "data_quality_limitations.csv", index=False)
data_quality_log
